[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MagedSaeed/instructions-tuning/blob/main/Notebooks/Experiments/Baselines/emotion_detection_tuning.ipynb)

# Get corekit from GitHub

coregit is a library of useful utilities that we can utilize for generation and tuning

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2,3"

In [2]:
from dotenv import load_dotenv
load_dotenv()

False

In [3]:
# !echo $GT_TOKEN # makesure the token is loaded

In [4]:
if 'jrcai_corekit' not in os.listdir('.'):
    !git clone https://${GT_TOKEN}@github.com/MagedSaeed/jrcai_corekit.git
else: # else, pull latest changes
    !cd jrcai_corekit && git pull && cd ..

Already up to date.


In [5]:
!pip install -r jrcai_corekit/requirements.txt

/bin/bash: /home/majed_alshaibani/Projects/instructions-tuning/venv/bin/pip: /home/majed_alshaibani/Projects/InstructionsTuning/venv/bin/python3: bad interpreter: No such file or directory


add jrcai_corekit to path

In [6]:
import sys
sys.path.append('jrcai_corekit/src')

check everything is working

In [7]:
from llm.text_generator import TextGenerator

/home/majed_alshaibani/Projects/instructions-tuning/venv/lib/python3.10/site-packages/transformers/deepspeed.py:24: FutureWarning: transformers.deepspeed module is deprecated and will be removed in a future version. Please import deepspeed modules directly from transformers.integrations
  warnings.warn(


# Constants

In [8]:
TAWJEEH_DATASET_NAME = 'opus-100'
HF_EXPERIMENTAL_DATASET_NAME = 'MagedSaeed/opus-100_ar_en_experimental'
TASK_NAME='machine_translation'
MODEL_PATH = "/hdd/shared_models/Meta-Llama-3.1-8B"

In [9]:
MODEL_NAME = MODEL_PATH.split('/')[-1]
TOKENIZER_PATH = MODEL_PATH

# Building the prompts dataset

In [10]:
import requests

from tqdm.auto import tqdm

prompts = None

tries = 10
for i in tqdm(range(tries)):
    api_response = requests.get(url='https://promptlab.up.railway.app/api/prompt/list?project_secret_key=6Wirj')
    if api_response.ok:
        prompts = api_response.json()
        break
if not prompts: raise Exception('Failed to fetch prompts')
prompts[:5]

  0%|          | 0/10 [00:00<?, ?it/s]

[{'id': 14870,
  'tags': [],
  'name': 'Review Classification-Basic Prompt most suitable',
  'task': {'name': 'review classification'},
  'status': 'APPROVED',
  'template': 'Task: Evaluate the satisfaction level of the given customer review and classify it on a scale from 1 to 5.\r\nOptions:\r\n1: Very Dissatisfied\r\n2: Dissatisfied\r\n3: Neutral\r\n4: Satisfied\r\n5: Very Satisfied\r\n\r\nCustomer Review:\r\n{{ text }}\r\nIndicate the most suitable satisfaction level (1, 2, 3, 4, or 5) based on the review. No need for extra explanation.\r\nAnswer: \r\n|||\r\n{{ label }}',
  'dataset_name': 'hard',
  'dataset_subset': 'plain_text',
  'answer_choices': ['1', '2', '3', '4', '5'],
  'text_direction': 'ltr'},
 {'id': 14869,
  'tags': [],
  'name': 'answer keyword at the end of the prompt',
  'task': {'name': 'multiple choice'},
  'status': 'APPROVED',
  'template': 'This is a question. Select the correct answer!\r\n\r\nQuestion: \r\n{{Question}}\r\n\r\nChoices:\r\n{% set choices = [A,B,C

In [11]:
len(prompts)

333

In [12]:
filtered_prompts = list(filter(lambda prompt: prompt['status'] == 'APPROVED' and prompt['text_direction'].lower() == 'ltr', prompts))
len(filtered_prompts)

221

## Finetuning

### Get the dataset prompts

In [13]:
# you can either filter by task or dataset
dataset_prompts = list(
    filter(
        lambda prompt: TAWJEEH_DATASET_NAME in prompt['dataset_name'],
        filtered_prompts,
    )
)
len(dataset_prompts)

10

In [14]:
SELECTED_PROMPTS_IDS = [
    14684,   
    14688,
    14680,
    14682,
    14640,
]

In [15]:
dataset_prompts = list(filter(lambda prompt: prompt['id'] in SELECTED_PROMPTS_IDS, filtered_prompts))
len(dataset_prompts)

5

### Download the dataset

In [16]:
import datasets

In [17]:
hf_exp_dataset = datasets.load_dataset(HF_EXPERIMENTAL_DATASET_NAME)
hf_exp_dataset

DatasetDict({
    train: Dataset({
        features: ['ar', 'en'],
        num_rows: 30000
    })
    test: Dataset({
        features: ['ar', 'en'],
        num_rows: 2000
    })
})

In [18]:
hf_exp_dataset = hf_exp_dataset.map(lambda example: {'translation':{'en':example['en'], 'ar':example['ar']}}, remove_columns=['en', 'ar'])
hf_exp_dataset

DatasetDict({
    train: Dataset({
        features: ['translation'],
        num_rows: 30000
    })
    test: Dataset({
        features: ['translation'],
        num_rows: 2000
    })
})

### Merge the prompts

In [19]:
from jinja2 import Environment, StrictUndefined

In [20]:
import re
def preprocess_template(template):
    # remove punc at the end
    prefix,suffix = template.split('|||')
    # remove multi spaces
    # prefix = re.sub(r'\s+', ' ', prefix)
    return f'{prefix.strip()}\n{suffix.strip()}' # output is always the last line!

In [21]:
def apply_template(prompt_template, sample):
    try:
        template = prompt_template['template']
        template = preprocess_template(template)
        env = Environment(undefined=StrictUndefined)
        template = env.from_string(template)
        rendered_template = template.render(**sample)
        return rendered_template
    except Exception as e:
        print(prompt_template)
        print(sample)
        raise e
        

see how the template is applied on different examples

### Perform prompt-merge on one example prompt, for experimentation

In [22]:
example_prompt_template = dataset_prompts[0]
print(apply_template(example_prompt_template, hf_exp_dataset['train'][3]))

Task: Carefully translate the following English sentence into Arabic, ensuring that the translation reflects the original intent and context.

English Sentence: Okay.

Guidelines:
1. Accurate Meaning: Ensure the translation conveys the exact meaning of the English sentence without losing any details.
2. Cultural and Contextual Fit: Consider any cultural or contextual nuances to translate feel natural in Arabic.
3. Tone Consistency: Maintain the same tone and style as the original, whether formal, casual, or neutral.
حسناً؟


In [23]:
step_size = len(hf_exp_dataset['train'])/len(dataset_prompts)
step_size

6000.0

In [24]:
rendered_train_prompts_dataset = list()
for i,sample in enumerate(tqdm(hf_exp_dataset['train'])):
    if i % step_size == 0:
        print(f'rending {dataset_prompts[int(i/step_size)]["template"]}','sample index:',i)
    rendered_train_prompts_dataset.append(
        apply_template(dataset_prompts[int(i/step_size)], sample)
    )
len(rendered_train_prompts_dataset)

  0%|          | 0/30000 [00:00<?, ?it/s]

rending Task: Carefully translate the following English sentence into Arabic, ensuring that the translation reflects the original intent and context.

English Sentence: {{ translation['en'] }}

Guidelines:
1. Accurate Meaning: Ensure the translation conveys the exact meaning of the English sentence without losing any details.
2. Cultural and Contextual Fit: Consider any cultural or contextual nuances to translate feel natural in Arabic.
3. Tone Consistency: Maintain the same tone and style as the original, whether formal, casual, or neutral.

|||
{{ translation['ar'] }} sample index: 0
rending Task: Translate the following English sentence into Arabic by reasoning step by step.

English Sentence: {{ translation['en'] }}

Step 1: Break Down the Meaning
Start by carefully analyzing the sentence. Identify the subject, verb, and key details to fully understand its meaning in context.

Step 2: Identify Important Words
Highlight the keywords or phrases that carry significant meaning, and thi

30000

## Finetune the LLM

In [25]:
GLOBAL_SEED = 42

In [26]:
import random
random.seed(GLOBAL_SEED)

In [27]:
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from llm import train_llm, LLMLoader, Llama3Initializer,LoRAConfigRepository
from sklearn.model_selection import train_test_split

In [28]:
llm_loader = LLMLoader(
        MODEL_PATH,
        llm_initializer=Llama3Initializer(),
)
llm_loader

In [29]:
model, tokenizer, generation_config = llm_loader()

loading configuration file /hdd/shared_models/Meta-Llama-3.1-8B/config.json
Model config LlamaConfig {
  "_name_or_path": "/hdd/shared_models/Meta-Llama-3.1-8B",
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 128000,
  "eos_token_id": 128001,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 14336,
  "max_position_embeddings": 131072,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "num_key_value_heads": 8,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_scaling": {
    "factor": 8.0,
    "high_freq_factor": 4.0,
    "low_freq_factor": 1.0,
    "original_max_position_embeddings": 8192,
    "rope_type": "llama3"
  },
  "rope_theta": 500000.0,
  "tie_word_embeddings": false,
  "torch_dtype": "bfloat16",
  "transformers_version": "4.45.2",
  "use_cache": true,
  "vocab_size": 128256
}


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

All model checkpoint weights were used when initializing LlamaForCausalLM.

All the weights of LlamaForCausalLM were initialized from the model checkpoint at /hdd/shared_models/Meta-Llama-3.1-8B.
If your task is similar to the task the model of the checkpoint was trained on, you can already use LlamaForCausalLM for predictions without further training.
loading configuration file /hdd/shared_models/Meta-Llama-3.1-8B/generation_config.json
Generate config GenerationConfig {
  "bos_token_id": 128000,
  "do_sample": true,
  "eos_token_id": 128001,
  "temperature": 0.6,
  "top_p": 0.9
}

loading file tokenizer.json
loading file tokenizer.model
loading file added_tokens.json
loading file special_tokens_map.json
loading file tokenizer_config.json
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
loading configuration file /hdd/shared_models/Meta-Llama-3.1-8B/generation_config.json
Generate config GenerationConfig {
  "bos_tok

In [30]:
import re

train_samples,eval_samples = train_test_split(
    rendered_train_prompts_dataset,
    test_size=0.1,
    random_state=GLOBAL_SEED,
)

def generate_tuple(sample):
    sample_lines = sample.splitlines()
    prefix = '\n'.join(sample_lines[:-1])
    prefix += '\nTranslation:'
    prefix = prefix.strip()
    suffix = sample_lines[-1].strip()
    suffix = f' {suffix}' # adding this space is important to split between input and output
    return prefix,suffix

train_samples = list(map(generate_tuple,train_samples))
eval_samples = list(map(generate_tuple,eval_samples))
len(train_samples), len(eval_samples), train_samples[:5], eval_samples[:5]

(27000,
 3000,
 [('Task: Carefully translate the following English sentence into Arabic, ensuring that the translation reflects the original intent and context.\n\nEnglish Sentence: Why do you ask?\n\nGuidelines:\n1. Accurate Meaning: Ensure the translation conveys the exact meaning of the English sentence without losing any details.\n2. Cultural and Contextual Fit: Consider any cultural or contextual nuances to translate feel natural in Arabic.\n3. Tone Consistency: Maintain the same tone and style as the original, whether formal, casual, or neutral.\nTranslation:',
   ' لمَ تسألان؟ لقد مات'),
  ("Task: Translate the following sentence from English to Arabic as accurately as possible.\nEnglish Sentence: If it's any consolation, the, uh... the guy stole my Wall Street Journal once.\n\nResponse:\nProvide the translation in Arabic.\nTranslation:",
   ' إن كان في هذا أي عزاء ٍ لكِ فإن ذلك الرجل قد سرق مني مجلة وول ستريت " في إحدى المرات "'),
  ('Task: Translate the following English sente

In [31]:
train_llm(
    model=model,
    tokenizer=tokenizer,
    train_samples=train_samples,
    eval_samples=eval_samples,
    peft_config=LoRAConfigRepository.llama_3(),
    learning_rate=2.5e-4,
    epochs_count=10,
    train_batch_size=4,
    eval_batch_size=4,
    output_dir=f'Notebooks/Experiments/{TASK_NAME}/{TAWJEEH_DATASET_NAME}/tuned_models/{MODEL_NAME}',
    early_stopping_patience=10,
    eval_steps=500,
)

peft config LoraConfig(peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, base_model_name_or_path=None, revision=None, task_type=<TaskType.CAUSAL_LM: 'CAUSAL_LM'>, inference_mode=False, r=16, target_modules={'q_proj', 'v_proj'}, lora_alpha=32, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', loftq_config={}, use_dora=False, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False))


/home/majed_alshaibani/Projects/instructions-tuning/venv/lib/python3.10/site-packages/transformers/training_args.py:1545: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
PyTorch: setting up devices
The default value for the training argument `--report_to` will change in v5 (from all installed integrations to none). In v5, you will need to use `--report_to all` to get the same behavior as now. You should start updating your code and make this info disappear :-).
You have loaded a model on multiple GPUs. `is_model_parallel` attribute will be force-set to `True` to avoid any unexpected behavior such as device placement mismatching.
Using auto half precision backend

***** Running Evaluation *****
  Num examples = 3000
  Batch size = 4


loading configuration file /hdd/shared_models/Meta-Llama-3.1-8B/config.json
Model config LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 128000,
  "eos_token_id": 128001,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 14336,
  "max_position_embeddings": 131072,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "num_key_value_heads": 8,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_scaling": {
    "factor": 8.0,
    "high_freq_factor": 4.0,
    "low_freq_factor": 1.0,
    "original_max_position_embeddings": 8192,
    "rope_type": "llama3"
  },
  "rope_theta": 500000.0,
  "tie_word_embeddings": false,
  "torch_dtype": "bfloat16",
  "transformers_version": "4.45.2",
  "use_cache": true,
  "vocab_size": 128256
}



{'eval_loss': 2.2277824878692627, 'eval_model_preparation_time': 0.0003, 'eval_runtime': 63.9786, 'eval_samples_per_second': 46.891, 'eval_steps_per_second': 11.723}


***** Running training *****
  Num examples = 27,000
  Num Epochs = 10
  Instantaneous batch size per device = 4
  Total train batch size (w. parallel, distributed & accumulation) = 4
  Gradient Accumulation steps = 1
  Total optimization steps = 67,500
  Number of trainable parameters = 6,815,744


...

{'eval_loss': 1.671685814857483, 'eval_model_preparation_time': 0.0003, 'eval_runtime': 53.3519, 'eval_samples_per_second': 56.23, 'eval_steps_per_second': 14.058, 'epoch': 2.6666666666666665}



***** Running Evaluation *****
  Num examples = 3000
  Batch size = 4


{'eval_loss': 1.6672662496566772, 'eval_model_preparation_time': 0.0003, 'eval_runtime': 53.3665, 'eval_samples_per_second': 56.215, 'eval_steps_per_second': 14.054, 'epoch': 2.7407407407407405}




Training completed. Do not forget to share your model on huggingface.co/models =)




1.6516389846801758

In [ ]:
exit()

: 